In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from dllm.core.schedulers import LinearAlphaScheduler
from dllm.core.samplers.mdlm import MDLMSampler, MDLMSamplerConfig


/home/prasanna/coding/diff-decoder/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'lm_eval'

In [ ]:
# 1. Initialize your custom/HuggingFace model and Tokenizer
# (Assuming the model is a Masked Diffusion LM like LLaDA)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForMaskedLM.from_pretrained("dllm-hub/Qwen3-0.6B-diffusion-bd3lm-v0.1", dtype=torch.bfloat16, trust_remote_code=True).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("dllm-hub/Qwen3-0.6B-diffusion-bd3lm-v0.1", trust_remote_code=True)

# Mock scheduler (often instantiated within your framework; MDLMSampler uses self.scheduler)
scheduler = LinearAlphaScheduler()

# 2. Configure the Sampler
config = MDLMSamplerConfig(
    max_new_tokens=64, 
    steps=64,              # Take 64 steps to fully unmask the generated text
    block_size=64,         # Process in blocks of 64
    temperature=0.1,       # Slight stochasticity
    cfg_scale=1.5,         # Classifier-free guidance multiplier
    remasking="low_confidence",
    return_dict=True       # Returns BaseSamplerOutput instead of pure tensor
)

In [ ]:
# 3. Instantiate the MDLMSampler
sampler = MDLMSampler(
    model=model, 
    tokenizer=tokenizer, 
    scheduler=scheduler
)

# ---------------------------------------------------------
# Scenario A: Standard Text Generation (Appending)
# ---------------------------------------------------------
prompts = ["The capital of France is"]
tokenized_inputs = [tokenizer.encode(p) for p in prompts]

output = sampler.sample(
    inputs=tokenized_inputs,
    config=config
)

print("Generation Result:")
print(tokenizer.decode(output.sequences[0]))

# ---------------------------------------------------------
# Scenario B: Text Infilling (Replacing masks in context)
# ---------------------------------------------------------
mask_id = tokenizer.mask_token_id
# Creating an input: "I went to the [MASK] [MASK] to buy some groceries."
infill_tokens = tokenizer.encode("I went to the ") + [mask_id, mask_id] + tokenizer.encode(" to buy some groceries.", add_special_tokens=False)

infill_output = sampler.infill(
    inputs=[infill_tokens],
    config=config
)

print("\nInfill Result:")
print(tokenizer.decode(infill_output.sequences[0]))